# Part 4: The agent (and then break it)

Eyes + ears + brain + mouth, together. The model is fixed. What you put in front
of it is yours, and that's what this notebook is about.

Every turn, three decisions:

| Decision | Options | Why it matters |
|---|---|---|
| **When to look** | `always` · `when_asked` · `never` · `before_after` · `burst` | every image is hundreds of tokens |
| **What to remember** | `latest_image` · `all_images` · `captions` | the prompt is the model's only memory |
| **What to hide** | a regex, e.g. `SECRET:.*` | state the bot needs but you shouldn't see |

A **mode** is a system prompt plus those three choices.

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

In [ ]:
from agent.brain import Agent
from agent.eyes import Eyes, preprocess, how_different, save
import modes

ORIGINAL_WANTS_TO_LOOK = Agent.wants_to_look   # so the Build challenges can be undone

CAMERA = config.CAMERA          # 0 = webcam, or a photo / folder path
eyes = Eyes(CAMERA)
nb.show(eyes.frame(), width=300)

## The modes

Edit anything here and re-run the cell, then re-run the panel cell.
`{now}` in a system prompt becomes the current time (the model has no clock).

In [ ]:
# Shared rules, used by most modes below. Edit these too.
VOICE_RULES = " Your reply is spoken aloud: one to three short sentences, no lists, no markdown, no emoji."
HONESTY = " If something isn't visible in the image, say you can't see it. Never guess text you can't read."

MODES = [
    dict(
        name="assistant",
        system="You are a friendly assistant that can see through the user's webcam. It is {now}."
        + HONESTY + VOICE_RULES,
        look="when_asked",
        memory="latest_image",
        max_turns=8,
    ),
    dict(
        name="grumpy art critic",
        system="You are a pompous, grumpy art critic. Everything the webcam shows is an 'installation' "
        "you are reviewing. Be theatrical and funny, never mean about the person themselves."
        + VOICE_RULES,
        look="always",
        memory="latest_image",
        max_turns=4,
        temperature=1.0,
    ),
    dict(
        name="sports commentator",
        system="You are an over-excited live sports commentator. Narrate what the person on camera is "
        "doing as if it's the final seconds of a championship. Exactly two sentences." + VOICE_RULES,
        look="burst",
        burst=3,
        burst_gap=0.6,
        memory="latest_image",
        max_turns=3,
        temperature=1.0,
    ),
    dict(
        name="i spy",
        system="You are playing I Spy with the user. On your first turn, silently pick ONE clearly "
        "visible object and write it on its own line as 'SECRET: <object>'. Then say 'I spy with my "
        "little eye something...' and give a colour or shape clue. On later turns, tell the user if "
        "their guess is right; if wrong, give a new small clue. Never reveal the secret unless they "
        "guess it or give up." + VOICE_RULES,
        look="always",
        memory="latest_image",
        kickoff="Start the game.",
        hide=r"(?im)^\s*SECRET:.*$",
        max_turns=12,
    ),
    dict(
        name="what changed?",
        system="Image 1 is the scene BEFORE. Image 2 is NOW. The user moved, added or removed something. "
        "Work out what changed. If nothing obvious changed, say so." + HONESTY + VOICE_RULES,
        look="before_after",
        memory="latest_image",
        max_turns=2,
        temperature=0.3,
    ),
    dict(
        name="charades",
        system="You are playing charades. The images are frames from a few seconds of video, in order. "
        "The user is acting something out. Guess what it is in one sentence, with confidence and flair."
        + VOICE_RULES,
        look="burst",
        burst=4,
        burst_gap=0.8,
        memory="latest_image",
        max_turns=6,
        temperature=0.9,
    ),
    dict(
        name="fridge chef",
        system="You are a practical home chef. Look at the ingredients or food the user shows you and "
        "suggest one simple thing to make with them. Mention only ingredients you can actually see, "
        "plus basic pantry staples." + VOICE_RULES,
        look="always",
        memory="captions",
        max_turns=6,
    ),
    dict(
        name="memory palace",
        system="You can see through the user's webcam. It is {now}. Earlier turns may include notes "
        "like [camera: ...] describing what you saw then. Use them to answer questions about what "
        "happened earlier, e.g. 'where did I put my keys?'." + HONESTY + VOICE_RULES,
        look="always",
        memory="captions",
        max_turns=20,
    ),
]

## Your mode

Make it yours. Ideas: plant doctor, reality-TV outfit judge, rubber-duck debugger
(point the camera at your screen), museum audio guide for your desk, translator
for any text you hold up.

In [ ]:
MY_MODE = dict(
    name="YOUR MODE",
    system="You are a pirate who can see through a webcam. Comment on what you see like it's treasure."
    + VOICE_RULES,
    look="always",            # always | when_asked | never | before_after | burst
    memory="latest_image",    # latest_image | all_images | captions
    max_turns=6,
    temperature=0.9,
)

ALL_MODES = MODES + [MY_MODE]
for i, m in enumerate(ALL_MODES, 1):
    print(f"{i}. {m['name']:<22} look={m.get('look'):<13} memory={m.get('memory')}")

## Peek: what does the model actually get?

Before playing, look at one prompt. Images are shown as placeholders so it fits
on screen. Try a question with "see" or "holding" in it, then one without.

In [ ]:
def peek(messages):
    for m in messages:
        c = m["content"]
        if isinstance(c, str):
            print(f"[{m['role']}] {c}")
        else:
            parts = [f"<image {len(p['image_url']['url']) * 3 // 4 // 1024} KB>" if p["type"] == "image_url"
                     else p.get("text", "") for p in c]
            print(f"[{m['role']}] " + " ".join(parts))
        print()

bot = Agent(llm, eyes, ALL_MODES[0])
said = "what am I holding?"
peek(bot.build_messages(said, bot.gather_images(said)))

## Play

Pick a mode, then **Start talking** / **Stop**, or type and press **Send**.
Switching modes wipes the conversation. **Reset** retakes the BEFORE photo in
*what changed?*.

Good first tries: the critic on your desk, charades (act, press Stop, keep acting:
it grabs frames right after), I Spy.

In [ ]:
bot = Agent(llm, eyes, ALL_MODES[0])
panel = nb.agent_panel(bot, ALL_MODES)

No mic, or the panel won't show? Same thing, typed:

In [ ]:
bot.set_mode(ALL_MODES[1])          # grumpy art critic
res = bot.turn("Review my desk, please.", on_token=nb.stream)
print("\n\n", res.reply.stats())

## Knob challenges (2 minutes each)

1. **Your mode.** Edit `MY_MODE`, re-run it and the panel.
2. **Hallucination hunt.** In mode 1 ask about things that aren't there: *"what's on the whiteboard
   behind me?"*. Edit `HONESTY` (top of the modes cell) until it admits it can't see them. How short can you make it?
3. **Memory.** In *memory palace*, put your keys in view, chat about something else for three turns,
   then ask where your keys are. Switch the **memory** dropdown and repeat. Which remembers? Which is fastest?
4. **Hidden state.** In the modes cell, delete the `hide=` line from I Spy. Now you can see its secret.
   Delete the SECRET instruction too, and watch it change its mind about what it picked.
5. **Temperature.** Critic at 0.2 vs 1.3.

---

## Build challenge 1: let the model decide when to look

`when_asked` uses a keyword list (`modes.LOOK_WORDS`), which is dumb: *"this is great"*
turns the camera on, *"is my hair a mess?"* doesn't. Replace it with a tiny extra
model call that answers YES or NO. What does it cost per turn?

In [ ]:
ROUTER_PROMPT = ("You decide if a voice assistant needs to look through the webcam to answer. "
                 "Answer with exactly one word: YES or NO.")

def needs_camera(self, said: str) -> bool:
    # TODO: ask self.llm with ROUTER_PROMPT + said (stream=False, max_tokens=3, temperature=0)
    #       and return True if the reply starts with "Y". Print how long it took.
    return ORIGINAL_WANTS_TO_LOOK(self, said)

Agent.wants_to_look = needs_camera
for q in ("is my hair a mess?", "this is great", "tell me a joke"):
    print(q, "->", bot.wants_to_look(q))

*Solution below, click `...` to reveal. To undo: `Agent.wants_to_look = ORIGINAL_WANTS_TO_LOOK`*

In [ ]:
# SOLUTION
import time

def needs_camera(self, said: str) -> bool:
    t0 = time.perf_counter()
    r = self.llm.chat([system(ROUTER_PROMPT), user(said)], stream=False, max_tokens=3, temperature=0)
    yes = r.text.strip().upper().startswith("Y")
    print(f"  router: {'LOOK' if yes else 'no look'} ({time.perf_counter() - t0:.2f}s)")
    return yes

Agent.wants_to_look = needs_camera
for q in ("is my hair a mess?", "this is great", "tell me a joke"):
    print(q, "->", bot.wants_to_look(q))

## Build challenge 2: speak up when something happens

So far it only talks when you do. This watches the camera and narrates when the
scene changes. Tweak `THRESHOLD`, `COOLDOWN` and the `NARRATOR` prompt (security
guard? over-excited dog?). Stop it with the ■ button.

In [ ]:
NARRATOR = ("You are a nature documentary narrator watching a webcam. Image 1 is a moment ago, image 2 is now. "
            "Something just changed. Narrate what happened in one hushed, dramatic sentence.")
THRESHOLD = 0.06     # 0-1, how much change counts
COOLDOWN = 8         # seconds of quiet after speaking
WATCH_FOR = 60       # seconds

import time
calm, last_spoke, t_end = eyes.frame(), 0.0, time.time() + WATCH_FOR
try:
    while time.time() < t_end:
        time.sleep(0.25)
        if not eyes.is_live:
            eyes.next_file()
        now = eyes.frame()
        diff = how_different(calm, now)
        print(f"\rchange {diff:.3f} {'#' * int(diff * 200):<40}", end="", flush=True)
        if diff < THRESHOLD or time.time() - last_spoke < COOLDOWN:
            if diff < THRESHOLD / 2:
                calm = now   # drift with slow lighting changes
            continue
        r = llm.chat([system(NARRATOR), user("What happened?", images=[preprocess(calm), preprocess(now)])],
                     stream=False, max_tokens=60)
        print("\nnarrator:", r.text.strip())
        mouth.speak(r.text)
        last_spoke, calm = time.time(), eyes.frame()
except KeyboardInterrupt:
    pass
print("\nstopped")

## Build challenge 3: give it hands

The model can only write text. But if we agree on a format, text can press buttons:

```
ACTION: save_note | buy more coffee
ACTION: take_photo | plant_day1
```

We hide those lines (same trick as I Spy's `SECRET:`) and run them. That's function
calling with a regex and an if-statement. Add your own action: a shopping list,
a timer, switching modes by voice.

In [ ]:
import os, re

NOTES = "notes.txt"
ACTION_RE = re.compile(r"(?im)^\s*ACTION:\s*(\w+)\s*\|\s*(.*)$")

def read_notes():
    return open(NOTES).read().strip() if os.path.exists(NOTES) else "(none yet)"

HANDS_MODE = dict(
    name="assistant with hands",
    system=("You can see through the user's webcam and you can do things by writing a line on its own:\n"
            "ACTION: save_note | <text to remember>\n"
            "ACTION: take_photo | <short_file_name>\n"
            "Use them only when the user asks you to remember something or take a picture. "
            "After any ACTION line, also say one short sentence confirming it.\n"
            "Current saved notes:\n{notes}") + VOICE_RULES,
    look="when_asked", memory="latest_image", max_turns=8,
    hide=r"(?im)^\s*ACTION:.*$",
)

class HandyAgent(Agent):
    def build_messages(self, said, images):
        msgs = super().build_messages(said, images)
        msgs[0]["content"] = msgs[0]["content"].replace("{notes}", read_notes())
        return msgs

    def turn(self, said, *args, **kwargs):
        res = super().turn(said, *args, **kwargs)
        last = self.turns[-1]
        for name, arg in ACTION_RE.findall(last.reply):
            arg = arg.strip()
            if name == "save_note":
                with open(NOTES, "a") as f:
                    f.write(f"- {arg}\n")
                print(f"[did] saved note: {arg}")
            elif name == "take_photo" and last.images:
                fname = re.sub(r"[^\w-]", "_", arg)[:40] + ".jpg"
                print("[did] saved", save(last.images[-1], fname))
            # TODO: elif name == "your_action": ...
            else:
                print(f"[skip] {name} {arg!r}")
        return res

modes.LOOK_WORDS = modes.LOOK_WORDS + ("picture", "photo")
handy = HandyAgent(llm, eyes, HANDS_MODE)
for said in ("remember that I need to buy coffee",
             "take a picture of this and call it desk",
             "what notes do I have?"):
    print("you:", said)
    res = handy.turn(said, on_token=nb.stream)
    print("\n")

Want it in the panel? `panel = nb.agent_panel(handy, [HANDS_MODE])`

## Show and tell

Last 5 minutes: the best mode in each row gets demoed to the room.

When you're done: `eyes.close()`

In [ ]:
eyes.close()